# What's in an Avocado Toast: A Supply Chain Analysis

![](avocado_wallpaper.jpeg)

You find yourself in London, crafting a delectable avocado toast, a dish that has risen dramatically in popularity on breakfast menus since the 2010s. This straightforward recipe requires just five ingredients: a ripe avocado, half a lemon, a generous pinch of salt flakes, two slices of sourdough bread, and a good drizzle of extra virgin olive oil. Most of these ingredients are now staples in grocery stores, and as you will find with this project, that is no small feat!

In this project, you'll conduct a supply chain analysis of three ingredients used in avocado toast using the Open Food Facts database. This database contains extensive, openly-sourced information on various foods, including their origins. Through this analysis, you will gain an in-depth understanding of the complex supply chain involved in producing a single dish.

Three pairs of files are provided in the data folder:
- A CSV file for each ingredient, such as `avocado.csv`, with data about each food item and countries of origin.
- A TXT file for each ingredient, such as `relevant_avocado_categories`, containing only the category tags of interest for that food.

Here are some other key points about these files:
- Some of the rows of data in each of the three CSV files do not contain relevant data for your investigation. In each dataset, you will need to filter out rows with irrelevant data, based on values in the `categories_tags` column. Examples of categories are fruits, vegetables, and fruit-based oils. Filter the DataFrame to include only rows where `categories_tags` contains one of the tags in the relevant categories for that ingredient.
- Each row of data usually has multiple category tags in the `categories_tags` column.
There is a column in each CSV file called `origins_tags`, which contains strings for the country of origin of each item.

After completing this project, you'll be armed with a list of ingredients and their countries of origin and be well-positioned to launch into other analyses that explore how long, on average, these ingredients spend at sea.

[Open Food Facts database](https://world.openfoodfacts.org/)

In [13]:
import pandas as pd

# 1. تعريف الدالة لتحليل بلد المنشأ الأكثر شيوعاً
def get_top_origin(df, category_file):
    # قراءة التصنيفات المطلوبة من الملف
    with open(category_file, 'r') as f:
        categories = f.read().splitlines()
    
    # قائمة الأعمدة المطلوبة فقط (Subsetting)
    relevant_cols = [
        'code', 'lc', 'product_name_en', 'quantity', 'serving_size', 
        'packaging_tags', 'brands', 'brands_tags', 'categories_tags', 
        'labels_tags', 'countries', 'countries_tags', 'origins', 'origins_tags'
    ]
    
    # تصفية الأعمدة المطلوبة
    df = df[relevant_cols]

    # تصفية البيانات لتشمل فقط المنتجات المباعة في المملكة المتحدة (شرط أساسي)
    # ملاحظة: في بيانات Open Food Facts، غالباً ما تكون القيمة 'en:united-kingdom'
    df = df[df['countries_tags'].str.contains('united-kingdom', na=False)]

    # تصفية الصفوف بناءً على التصنيفات (Categories)
    df = df[df['categories_tags'].str.contains('|'.join(categories), na=False)]

    # التعامل مع عمود بلد المنشأ (origins_tags)
    df = df.dropna(subset=['origins_tags'])
    
    # تقسيم القيم إذا كان هناك أكثر من بلد منشأ وتوسيع الإطار
    df['origins_tags'] = df['origins_tags'].str.split(',')
    df = df.explode('origins_tags')

    # تنظيف أسماء الدول (إزالة البادئات en:، الرموز، والشرطات)
    # نستخدم regex لإبقاء الحروف والمسافات فقط كما طلب السؤال
    df['origins_tags'] = df['origins_tags'].str.replace('en:', '', case=False)
    df['origins_tags'] = df['origins_tags'].str.replace('-', ' ')
    df['origins_tags'] = df['origins_tags'].str.replace(r'[^a-zA-Z\s]', '', regex=True).str.strip()

    # إعادة اسم الدولة الأكثر تكراراً
    return df['origins_tags'].value_counts().idxmax()

# 2. تحميل البيانات (تأكد من المسارات الصحيحة للملفات في بيئة العمل لديك)
avocado = pd.read_csv('data/avocado.csv', sep='\t')
olive_oil = pd.read_csv('data/olive_oil.csv', sep='\t')
sourdough = pd.read_csv('data/sourdough.csv', sep='\t')

# 3. استخراج النتائج وتخزينها في المتغيرات المطلوبة
top_avocado_origin = get_top_origin(avocado, 'data/relevant_avocado_categories.txt')
top_olive_oil_origin = get_top_origin(olive_oil, 'data/relevant_olive_oil_categories.txt')
top_sourdough_origin = get_top_origin(sourdough, 'data/relevant_sourdough_categories.txt')

# طباعة النتائج للتحقق
print(f"Top Avocado Origin: {top_avocado_origin}")
print(f"Top Olive Oil Origin: {top_olive_oil_origin}")
print(f"Top Sourdough Origin: {top_sourdough_origin}")


Top Avocado Origin: peru
Top Olive Oil Origin: greece
Top Sourdough Origin: united kingdom
